In [ ]:
#| default_exp preview

# Preview

> File preview extraction — returns the first N non-empty lines from text, PDF, DOCX, XLSX files, or an image marker.

In [ ]:
#| export
from pathlib import Path
from dataclasses import dataclass
import subprocess, shutil, zipfile

In [ ]:
#| export
@dataclass
class PreviewResult:
    """Result of a file preview extraction."""
    lines: list[str]
    is_image: bool = False
    error: str | None = None

In [ ]:
#| export
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.gif', '.webp', '.bmp', '.svg', '.ico', '.tiff', '.tif'}
TEXT_EXTS = {'.txt', '.md', '.markdown', '.rst', '.csv', '.tsv', '.log', '.json', '.yaml', '.yml',
             '.toml', '.ini', '.cfg', '.conf', '.xml', '.html', '.htm', '.css', '.js', '.ts',
             '.py', '.sh', '.bash', '.zsh', '.ps1', '.bat', '.cmd', '.sql', '.r', '.jl'}

In [ ]:
#| export
def _take_lines(text: str, n: int) -> list[str]:
    """Extract first `n` non-empty lines from `text`."""
    lines = []
    for line in text.splitlines():
        stripped = line.strip()
        if stripped:
            lines.append(stripped)
            if len(lines) >= n: break
    return lines

In [ ]:
#| export
def _preview_text(path: Path, n: int) -> PreviewResult:
    """Preview a text-based file."""
    try:
        text = path.read_text(encoding='utf-8', errors='replace')
        return PreviewResult(lines=_take_lines(text, n))
    except Exception as e:
        return PreviewResult(lines=[], error=str(e))

def _preview_pdf(path: Path, n: int) -> PreviewResult:
    """Preview a PDF using pdfplumber. A scanned PDF (no text layer) gets an explicit
    marker instead of being reported as an empty file — its pages are photos, not blanks."""
    try:
        import pdfplumber
        lines = []
        has_images = False
        with pdfplumber.open(path) as pdf:
            page_count = len(pdf.pages)
            for page in pdf.pages:
                text = page.extract_text() or ""
                if page.images: has_images = True
                for line in text.splitlines():
                    stripped = line.strip()
                    if stripped:
                        lines.append(stripped)
                        if len(lines) >= n: return PreviewResult(lines=lines)
        if not lines and has_images:
            noun = "page" if page_count == 1 else "pages"
            return PreviewResult(lines=[f"📷 Scanned PDF — {page_count} {noun}, no text layer", "Open with [o] key"], is_image=True)
        return PreviewResult(lines=lines)
    except Exception as e:
        return PreviewResult(lines=[], error=str(e))

def _preview_docx(path: Path, n: int) -> PreviewResult:
    """Preview a DOCX using python-docx."""
    try:
        from docx import Document
        doc = Document(path)
        lines = []
        for para in doc.paragraphs:
            text = para.text.strip()
            if text:
                lines.append(text)
                if len(lines) >= n: break
        return PreviewResult(lines=lines)
    except Exception as e:
        return PreviewResult(lines=[], error=str(e))

def _preview_epub(path: Path, n: int) -> PreviewResult:
    """Preview an EPUB by converting to markdown via pandoc — first `n` non-empty lines."""
    if not shutil.which('pandoc'):
        return PreviewResult(lines=[], error="pandoc not installed (apt install pandoc)")
    try:
        res = subprocess.run(['pandoc', '-f', 'epub', '-t', 'markdown', str(path)],
                             capture_output=True, text=True, timeout=30)
        if res.returncode != 0: return PreviewResult(lines=[], error=res.stderr.strip() or 'pandoc failed')
        return PreviewResult(lines=_take_lines(res.stdout, n))
    except Exception as e:
        return PreviewResult(lines=[], error=str(e))

def _preview_xlsx(path: Path, n: int) -> PreviewResult:
    """Preview an XLSX — first sheet, first N non-empty rows as tab-separated."""
    try:
        from openpyxl import load_workbook
        wb = load_workbook(path, read_only=True, data_only=True)
        ws = wb.active
        lines = []
        for row in ws.iter_rows(values_only=True):
            vals = [str(c) if c is not None else "" for c in row]
            line = "\t".join(vals).strip()
            if line:
                lines.append(line)
                if len(lines) >= n: break
        wb.close()
        return PreviewResult(lines=lines)
    except Exception as e:
        return PreviewResult(lines=[], error=str(e))

In [ ]:
#| export
def _preview_zip(path: Path, n: int) -> PreviewResult:
    """Preview a ZIP archive — list its entries (name and size) instead of garbled binary text."""
    try:
        with zipfile.ZipFile(path) as zf:
            infos = zf.infolist()
            lines = [f"{i.filename}  ({i.file_size:,} B)" for i in infos[:n]]
            if len(infos) > n: lines.append(f"... and {len(infos) - n} more")
            return PreviewResult(lines=lines or ["(empty archive)"])
    except Exception as e:
        return PreviewResult(lines=[], error=str(e))

In [ ]:
#| export
def get_preview(path: Path, n: int = 20) -> PreviewResult:
    """Get a preview of `path` — dispatches based on file extension."""
    suffix = path.suffix.lower()
    if suffix in IMAGE_EXTS:
        return PreviewResult(lines=[f"📷 Image: {path.name}", f"Open with [o] key"], is_image=True)
    if suffix == '.pdf': return _preview_pdf(path, n)
    if suffix == '.docx': return _preview_docx(path, n)
    if suffix == '.epub': return _preview_epub(path, n)
    if suffix in ('.xlsx', '.xls'): return _preview_xlsx(path, n)
    if suffix in ('.zip', '.jar'): return _preview_zip(path, n)
    # Default: try as text
    return _preview_text(path, n)

## Tests

In [ ]:
from fastcore.test import *
import tempfile, os

# Test text preview
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    f.write("\n\nLine 1\n\nLine 2\nLine 3\n\n")
    f.flush()
    result = get_preview(Path(f.name), n=2)
    test_eq(result.lines, ["Line 1", "Line 2"])
    test_eq(result.is_image, False)
    test_eq(result.error, None)
    os.unlink(f.name)

# Test empty file
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    f.write("")
    f.flush()
    result = get_preview(Path(f.name))
    test_eq(result.lines, [])
    test_eq(result.error, None)
    os.unlink(f.name)

# Test image detection
result = get_preview(Path("photo.jpg"))
test_eq(result.is_image, True)
assert "Image" in result.lines[0]

result = get_preview(Path("screenshot.PNG"))
test_eq(result.is_image, True)

# Test markdown file (treated as text)
with tempfile.NamedTemporaryFile(mode='w', suffix='.md', delete=False) as f:
    f.write("# Title\n\nSome content\n\n## Section\n")
    f.flush()
    result = get_preview(Path(f.name), n=3)
    test_eq(result.lines, ["# Title", "Some content", "## Section"])
    os.unlink(f.name)

# Test _take_lines helper
test_eq(_take_lines("a\n\nb\nc", 2), ["a", "b"])
test_eq(_take_lines("", 5), [])
test_eq(_take_lines("\n\n\n", 5), [])

print("\u2713 All preview tests passed")

In [ ]:
# Test zip preview lists entries instead of garbled binary
with tempfile.TemporaryDirectory() as tmp:
    zpath = Path(tmp) / "archive.zip"
    with zipfile.ZipFile(zpath, "w") as zf:
        zf.writestr("readme.txt", "hello world")
        zf.writestr("data/values.csv", "a,b,c\n1,2,3\n")

    result = get_preview(zpath)
    test_eq(result.error, None)
    test_eq(result.is_image, False)
    test_eq(len(result.lines), 2)
    assert "readme.txt" in result.lines[0]
    assert "data/values.csv" in result.lines[1]

    # Truncation: cap at n entries, note how many more
    result_capped = get_preview(zpath, n=1)
    test_eq(len(result_capped.lines), 2)
    assert "readme.txt" in result_capped.lines[0]
    assert "1 more" in result_capped.lines[1]

# Test empty zip
with tempfile.TemporaryDirectory() as tmp:
    zpath = Path(tmp) / "empty.zip"
    with zipfile.ZipFile(zpath, "w"): pass
    result = get_preview(zpath)
    test_eq(result.lines, ["(empty archive)"])

# Test corrupt zip surfaces an error instead of crashing
with tempfile.NamedTemporaryFile(suffix=".zip", delete=False) as f:
    f.write(b"not a real zip file")
    f.flush()
    result = get_preview(Path(f.name))
    assert result.error is not None
    os.unlink(f.name)

print("✓ All zip preview tests passed")

In [ ]:
# Test scanned PDF (image-only, no text layer) gets an explicit marker, not "(empty file)"
def _make_image_only_pdf(path: Path):
    """Minimal hand-built one-page PDF: one raw RGB image XObject, no text operators at all."""
    img_data = bytes([255, 0, 0, 0, 255, 0, 0, 0, 255, 255, 255, 0])  # 2x2 px RGB
    content = b"q 100 0 0 100 0 0 cm /Im0 Do Q"
    objects = [
        b"<< /Type /Catalog /Pages 2 0 R >>",
        b"<< /Type /Pages /Kids [3 0 R] /Count 1 >>",
        b"<< /Type /Page /Parent 2 0 R /Resources << /XObject << /Im0 5 0 R >> >> "
        b"/MediaBox [0 0 100 100] /Contents 4 0 R >>",
        b"<< /Length " + str(len(content)).encode() + b" >>\nstream\n" + content + b"\nendstream",
        b"<< /Type /XObject /Subtype /Image /Width 2 /Height 2 /ColorSpace /DeviceRGB "
        b"/BitsPerComponent 8 /Length " + str(len(img_data)).encode() + b" >>\nstream\n" + img_data + b"\nendstream",
    ]
    out = bytearray(b"%PDF-1.4\n")
    offsets = []
    for i, obj in enumerate(objects, start=1):
        offsets.append(len(out))
        out += f"{i} 0 obj\n".encode() + obj + b"\nendobj\n"
    xref_offset = len(out)
    n = len(objects) + 1
    out += f"xref\n0 {n}\n".encode() + b"0000000000 65535 f \n"
    for off in offsets: out += f"{off:010d} 00000 n \n".encode()
    out += f"trailer\n<< /Size {n} /Root 1 0 R >>\nstartxref\n{xref_offset}\n%%EOF".encode()
    path.write_bytes(bytes(out))

with tempfile.TemporaryDirectory() as tmp:
    ppath = Path(tmp) / "scan.pdf"
    _make_image_only_pdf(ppath)
    result = get_preview(ppath)
    test_eq(result.error, None)
    test_eq(result.is_image, True)
    assert "Scanned PDF" in result.lines[0]
    assert "1 page" in result.lines[0]

print("✓ Scanned PDF preview test passed")